# SleepWatch — Train & Export
Run all cells. Downloads `sleep_scorer.h` at the end. Drop that file into your Arduino sketch folder.

In [4]:
import numpy as np
import m2cgen as m2c
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
np.random.seed(42)

# ─────────────────────────────────────────────────────────────
#  DATASET GENERATION
#  Each sample simulates one completed test session.
#  alertness: 0.0 = crashed, 1.0 = fully awake
# ─────────────────────────────────────────────────────────────
N = 6000

def simulate(alertness):
    a = alertness
    n = lambda mu, sd: np.clip(np.random.normal(mu, sd), 0, 1)

    # --- Accuracy per round (0.0–1.0)
    # Sleepy brains lose accuracy faster on harder rounds
    r1_acc = n(0.95*a + 0.05,        0.07)
    r2_acc = n(0.88*a + 0.04,        0.09)
    r3_acc = n(0.80*a - 0.02,        0.11)  # hardest round, biggest hit

    # --- Speed per round (ms). Alert=fast, Sleepy=slow.
    # Alert baseline ~1500ms, sleepy baseline ~9000ms
    spd_base = 1500 + (1 - a) * 7500
    r1_speed = np.clip(np.random.normal(spd_base * 0.90, 400), 500, 15000)
    r2_speed = np.clip(np.random.normal(spd_base * 1.00, 600), 500, 15000)
    r3_speed = np.clip(np.random.normal(spd_base * 1.15, 800), 500, 15000)  # fatigue sets in

    # --- Trends (round3 - round1): negative = degrading
    speed_trend = r3_speed - r1_speed   # positive = slowing down = bad
    acc_trend   = r3_acc   - r1_acc     # negative = dropping accuracy = bad

    # --- Response variance: KEY feature.
    # Alert person: tight cluster. Sleepy person: micro-sleeps cause spikes.
    # We sample 9 individual response times and take their std dev.
    all_times = []
    for base_spd in [r1_speed, r2_speed, r3_speed]:
        for _ in range(3):
            # Sleepy people occasionally zone out (micro-sleep spike)
            spike = np.random.choice([1, 3], p=[0.85 + 0.14*a, 0.15 - 0.14*a])
            t = np.random.normal(base_spd * spike, 300 + (1-a)*1200)
            all_times.append(np.clip(t, 200, 15000))
    response_variance = np.std(all_times)   # raw ms std dev

    # --- Ground truth sleep score (0–100)
    # Weighted formula — the model will learn finer interactions
    acc_score  = (r1_acc*0.20 + r2_acc*0.30 + r3_acc*0.50) * 55
    spd_score  = (1 - np.clip(spd_base / 15000, 0, 1)) * 25
    var_pen    = np.clip(response_variance / 5000, 0, 1) * 10
    trend_pen  = np.clip(speed_trend / 5000, 0, 1) * 5 + np.clip(-acc_trend, 0, 1) * 5
    score = np.clip(acc_score + spd_score - var_pen - trend_pen + np.random.normal(0, 2), 0, 100)

    return [
        r1_acc, r2_acc, r3_acc,
        r1_speed, r2_speed, r3_speed,
        speed_trend, acc_trend,
        response_variance,
        score
    ]

# Sample realistic alertness distribution (beta — most people are somewhere in the middle)
alertness = np.random.beta(1.8, 1.8, N)
data = np.array([simulate(a) for a in alertness])

FEATURES = ['r1_acc','r2_acc','r3_acc',
            'r1_speed','r2_speed','r3_speed',
            'speed_trend','acc_trend',
            'response_variance']

X = data[:, :9].astype(np.float32)
y = data[:,  9].astype(np.float32)

print(f'Dataset: {X.shape}  Score range: {y.min():.1f}–{y.max():.1f}  Mean: {y.mean():.1f}')

Dataset: (6000, 9)  Score range: 0.0–76.7  Mean: 34.4


In [5]:
# ─────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

model = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae   = mean_absolute_error(y_test, preds)
print(f'Test MAE: {mae:.2f} points  (on a 0–100 scale)')

# Sanity checks
cases = [
    ('Fully awake',   0.95),
    ('Mildly tired',  0.70),
    ('Moderately sleepy', 0.45),
    ('Very sleepy',   0.20),
    ('Crashed',       0.05),
]
print(f'\n{"Case":<22} {"Score":>8}')
print('-'*32)
for label, al in cases:
    row = np.array(simulate(al)[:9], dtype=np.float32).reshape(1,-1)
    s   = np.clip(model.predict(row)[0], 0, 100)
    print(f'{label:<22} {s:>8.1f}')

Test MAE: 2.39 points  (on a 0–100 scale)

Case                      Score
--------------------------------
Fully awake                53.3
Mildly tired               45.7
Moderately sleepy          24.2
Very sleepy                14.6
Crashed                     6.9


In [6]:
# ─────────────────────────────────────────────────────────────
#  EXPORT TO C  via m2cgen
# ─────────────────────────────────────────────────────────────
raw_c = m2c.export_to_c(model, function_name='score_raw')

header = """// sleep_scorer.h
// Auto-generated by m2cgen from a GradientBoostingRegressor.
// Drop this file into your Arduino sketch folder. No other files needed.
#pragma once
#include <math.h>

// Feature order expected by score_session():
//   [0] r1_acc          – round 1 accuracy  (0.0–1.0)
//   [1] r2_acc          – round 2 accuracy  (0.0–1.0)
//   [2] r3_acc          – round 3 accuracy  (0.0–1.0)
//   [3] r1_speed        – round 1 avg response time (ms)
//   [4] r2_speed        – round 2 avg response time (ms)
//   [5] r3_speed        – round 3 avg response time (ms)
//   [6] speed_trend     – r3_speed - r1_speed  (positive = slowing = bad)
//   [7] acc_trend       – r3_acc   - r1_acc    (negative = dropping  = bad)
//   [8] response_variance – std dev of all 9 response times (ms)

"""

footer = """
// Clamp raw output and return final 0-100 sleep score.
static inline float score_session(double* input) {
    double raw = score_raw(input);
    if (raw < 0.0)   raw = 0.0;
    if (raw > 100.0) raw = 100.0;
    return (float)raw;
}
"""

full_header = header + raw_c + footer

with open('sleep_scorer.h', 'w') as f:
    f.write(full_header)

print(f'sleep_scorer.h written — {len(full_header):,} bytes')
print('Preview of generated C code:')
print('\n'.join(raw_c.split('\n')[:20]), '...')

sleep_scorer.h written — 237,276 bytes
Preview of generated C code:
double score_raw(double * input) {
    double var0;
    if (input[3] <= 4603.429443359375) {
        if (input[2] <= 0.5439852774143219) {
            if (input[1] <= 0.592702716588974) {
                if (input[2] <= 0.3564035892486572) {
                    if (input[8] <= 1143.9803466796875) {
                        var0 = 35.56671232926218;
                    } else {
                        var0 = 29.124891699406138;
                    }
                } else {
                    if (input[8] <= 1237.3184814453125) {
                        var0 = 41.54082227556893;
                    } else {
                        var0 = 36.64101720876731;
                    }
                }
            } else {
                if (input[2] <= 0.4727887362241745) { ...
